# Market Access RWE Generator

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Generar real-world evidence** using SQL aggregations
2. **Calculate outcomes metrics** (adherence, persistence)
3. **Analyze treatment patterns** with sequence analysis
4. **Compute cost-effectiveness** ratios
5. **Build payer evidence** packages

---

## What You'll Build

A **real-world evidence system** that generates payer-ready analyses:
- Clinical outcomes analysis (adherence, persistence, discontinuation)
- Treatment pattern identification
- Cost-effectiveness calculations
- Comparative effectiveness research
- Evidence packages for payer negotiations

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `COUNT() FILTER(WHERE)` - Conditional aggregations
- `LAG()` - Identify treatment switches
- `DATEDIFF()` - Calculate persistence days
- `PERCENTILE_CONT()` - Median and quartile calculations
- `GROUP BY GROUPING SETS` - Multi-dimensional aggregations

Let's begin!

---

## Paso 1: Configuración del Entorno

### Real-World Evidence (RWE)

RWE uses data from actual clinical practice (not controlled trials) to demonstrate:
- **Effectiveness**: Does it work in real patients?
- **Safety**: Real-world adverse events
- **Economic value**: Cost per quality-adjusted life year (QALY)
- **Comparative effectiveness**: vs. standard of care

### Why RWE Matters

Payers and formulary committees require RWE to:
- Make coverage decisions
- Set tier placement
- Negotiate rebates
- Demonstrate value for money

In [ ]:
CREATE DATABASE IF NOT EXISTS MARKET_ACCESS_DB;
CREATE SCHEMA IF NOT EXISTS MARKET_ACCESS_DB.PUBLIC;
USE SCHEMA MARKET_ACCESS_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db, CURRENT_SCHEMA() as schema, CURRENT_WAREHOUSE() as wh;

---

## Paso 2: Define Data Structure

### Tables

1. **`patients`** - Demographics, enrollment, index date
2. **`clinical_outcomes`** - Lab results, vitals over time
3. **`medication_fills`** - Prescriptions and refills
4. **`healthcare_utilization`** - ER, hospitalizations, office visits
5. **`costs`** - Medical and pharmacy spending

### Key Metrics

- **INR stability**: Primary anticoagulation outcome
- **PDC (Proportion of Days Covered)**: Adherence measure
- **Persistence**: Time on therapy
- **PMPM (Per Member Per Month)**: Cost metric

In [ ]:
-- Patient cohort
CREATE OR REPLACE TABLE patients (
    patient_id VARCHAR(50) PRIMARY KEY,
    age INTEGER,
    gender VARCHAR(10),
    insurance_type VARCHAR(50),
    index_drug VARCHAR(100),  -- Treatment being studied
    index_date DATE,  -- Treatment start date
    baseline_a1c FLOAT,
    baseline_weight_kg FLOAT,
    comorbidities ARRAY
);

-- Clinical outcomes over time
CREATE OR REPLACE TABLE clinical_outcomes (
    outcome_id VARCHAR(50) PRIMARY KEY,
    patient_id VARCHAR(50),
    measurement_date DATE,
    measurement_type VARCHAR(50),
    measurement_value FLOAT,
    days_from_index INTEGER
);

-- Medication fills
CREATE OR REPLACE TABLE medication_fills (
    fill_id VARCHAR(50) PRIMARY KEY,
    patient_id VARCHAR(50),
    fill_date DATE,
    drug_name VARCHAR(100),
    days_supply INTEGER,
    cost_usd DECIMAL(10,2)
);

-- Healthcare utilization
CREATE OR REPLACE TABLE healthcare_utilization (
    visit_id VARCHAR(50) PRIMARY KEY,
    patient_id VARCHAR(50),
    visit_date DATE,
    visit_type VARCHAR(50),  -- ER, inpatient, office_visit
    primary_diagnosis VARCHAR(100),
    cost_usd DECIMAL(10,2)
);

-- Cost tracking
CREATE OR REPLACE TABLE costs (
    cost_id VARCHAR(50) PRIMARY KEY,
    patient_id VARCHAR(50),
    cost_month DATE,
    medical_cost_usd DECIMAL(10,2),
    pharmacy_cost_usd DECIMAL(10,2)
);

SELECT 'Tables created!' as status;

---

## Paso 3: Generar Datos Sintéticos Patient Data

Creating realistic RWE data:
- **1,000 patients** on Xarelto vs. 500 on standard of care
- **12 months** of follow-up data
- Realistic outcome patterns (INR stability, bleeding event reduction)
- Variable adherence (some stop treatment)

In [ ]:
-- Generar patient cohort (1000 Xarelto, 500 standard of care)
INSERT INTO patients
SELECT 
    'PAT' || LPAD(SEQ4(), 5, '0') as patient_id,
    FLOOR(UNIFORM(35, 75, RANDOM())) as age,
    CASE WHEN UNIFORM(0,1,RANDOM()) > 0.5 THEN 'Female' ELSE 'Male' END as gender,
    CASE FLOOR(UNIFORM(1,4,RANDOM()))
        WHEN 1 THEN 'Commercial'
        WHEN 2 THEN 'Medicare'
        ELSE 'Medicaid'
    END as insurance_type,
    CASE WHEN SEQ4() <= 1000 THEN 'Xarelto' ELSE 'Warfarin' END as index_drug,
    DATEADD('month', -FLOOR(UNIFORM(6,18,RANDOM())), CURRENT_DATE()) as index_date,
    UNIFORM(7.5, 10.5, RANDOM())::DECIMAL(3,1) as baseline_a1c,
    UNIFORM(75, 120, RANDOM())::DECIMAL(5,1) as baseline_weight_kg,
    ARRAY_CONSTRUCT(
        CASE WHEN UNIFORM(0,1,RANDOM()) > 0.6 THEN 'Hypertension' END,
        CASE WHEN UNIFORM(0,1,RANDOM()) > 0.7 THEN 'Hyperlipidemia' END,
        CASE WHEN UNIFORM(0,1,RANDOM()) > 0.8 THEN 'Obesity' END
    ) as comorbidities
FROM TABLE(GENERATOR(ROWCOUNT => 1500));

-- Generar clinical outcomes (INR, bleeding events)
INSERT INTO clinical_outcomes
WITH month_mapping AS (
    SELECT CASE ROW_NUMBER() OVER (ORDER BY SEQ4())
        WHEN 1 THEN 0
        WHEN 2 THEN 3
        WHEN 3 THEN 6
        WHEN 4 THEN 12
    END as month_num
    FROM TABLE(GENERATOR(ROWCOUNT => 4))
),
patient_months AS (
    SELECT p.patient_id, p.index_date, p.index_drug, p.baseline_a1c, p.baseline_weight_kg,
           m.month_num
    FROM patients p
    CROSS JOIN month_mapping m
),
a1c_measurements AS (
    SELECT 
        UUID_STRING() as outcome_id,
        patient_id,
        DATEADD('month', month_num, index_date) as measurement_date,
        'A1C' as measurement_type,
        CASE 
            WHEN index_drug = 'Xarelto' THEN 
                GREATEST(5.5, baseline_a1c - (month_num * 0.3) + UNIFORM(-0.2,0.2,RANDOM()))
            ELSE 
                GREATEST(6.5, baseline_a1c - (month_num * 0.15) + UNIFORM(-0.2,0.2,RANDOM()))
        END::DECIMAL(3,1) as measurement_value,
        month_num * 30 as days_from_index
    FROM patient_months
),
weight_measurements AS (
    SELECT 
        UUID_STRING() as outcome_id,
        patient_id,
        DATEADD('month', month_num, index_date) as measurement_date,
        'Weight_kg' as measurement_type,
        CASE 
            WHEN index_drug = 'Xarelto' THEN 
                baseline_weight_kg * (1 - month_num * 0.02) + UNIFORM(-1,1,RANDOM())
            ELSE 
                baseline_weight_kg * (1 - month_num * 0.005) + UNIFORM(-1,1,RANDOM())
        END::DECIMAL(5,1) as measurement_value,
        month_num * 30 as days_from_index
    FROM patient_months
)
SELECT * FROM a1c_measurements
UNION ALL
SELECT * FROM weight_measurements;

-- Generar medication fills (varying adherence)
INSERT INTO medication_fills
WITH fill_schedule AS (
    SELECT p.patient_id, p.index_date, p.index_drug,
           f.fill_num,
           CASE WHEN index_drug = 'Xarelto' THEN 30 ELSE 90 END as days_supply,
           CASE WHEN index_drug = 'Xarelto' THEN 850 ELSE 25 END as cost
    FROM patients p
    CROSS JOIN (SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) as fill_num 
                FROM TABLE(GENERATOR(ROWCOUNT => 15))) f
    -- Some patients stop early (non-persistent)
    WHERE fill_num <= FLOOR(UNIFORM(3, 13, RANDOM()))
)
SELECT 
    UUID_STRING() as fill_id,
    patient_id,
    DATEADD('day', (fill_num - 1) * days_supply + FLOOR(UNIFORM(-5,5,RANDOM())), index_date) as fill_date,
    index_drug as drug_name,
    days_supply,
    cost::DECIMAL(10,2) as cost_usd
FROM fill_schedule;

-- Generar healthcare utilization (ER, hospital, office visits)
INSERT INTO healthcare_utilization
SELECT 
    UUID_STRING() as visit_id,
    p.patient_id,
    DATEADD('day', FLOOR(UNIFORM(0,365,RANDOM())), p.index_date) as visit_date,
    CASE FLOOR(UNIFORM(1,11,RANDOM()))
        WHEN 1 THEN 'ER'
        WHEN 2 THEN 'Inpatient'
        ELSE 'Office_Visit'
    END as visit_type,
    CASE FLOOR(UNIFORM(1,6,RANDOM()))
        WHEN 1 THEN 'Anticoagulation_Management'
        WHEN 2 THEN 'Hypoglycemia'
        WHEN 3 THEN 'Cardiovascular'
        ELSE 'Routine_Checkup'
    END as primary_diagnosis,
    CASE visit_type
        WHEN 'ER' THEN UNIFORM(500, 2000, RANDOM())
        WHEN 'Inpatient' THEN UNIFORM(5000, 15000, RANDOM())
        ELSE UNIFORM(100, 300, RANDOM())
    END::DECIMAL(10,2) as cost_usd
FROM patients p,
TABLE(GENERATOR(ROWCOUNT => 3))
WHERE UNIFORM(0,1,RANDOM()) > 0.3;

-- Generar monthly costs
INSERT INTO costs
WITH month_sequence AS (
    SELECT (ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1) as month_offset
    FROM TABLE(GENERATOR(ROWCOUNT => 12))
)
SELECT 
    UUID_STRING() as cost_id,
    p.patient_id,
    DATE_TRUNC('month', DATEADD('month', ms.month_offset, p.index_date)) as cost_month,
    UNIFORM(200, 800, RANDOM())::DECIMAL(10,2) as medical_cost_usd,
    CASE WHEN p.index_drug = 'Xarelto' THEN UNIFORM(700, 900, RANDOM()) 
         ELSE UNIFORM(20, 40, RANDOM()) END::DECIMAL(10,2) as pharmacy_cost_usd
FROM patients p
CROSS JOIN month_sequence ms;

-- Verify data
SELECT 
    (SELECT COUNT(*) FROM patients) as patients,
    (SELECT COUNT(*) FROM clinical_outcomes) as outcomes,
    (SELECT COUNT(*) FROM medication_fills) as fills,
    (SELECT COUNT(*) FROM healthcare_utilization) as visits;

In [ ]:
-- DIAGNOSTIC: Check data quality
SELECT 'Patients' as table_name, COUNT(*) as row_count FROM patients
UNION ALL
SELECT 'Clinical Outcomes', COUNT(*) FROM clinical_outcomes
UNION ALL
SELECT 'Clinical Outcomes (>= 180 days)', COUNT(*) FROM clinical_outcomes WHERE days_from_index >= 180
UNION ALL
SELECT 'Medication Fills', COUNT(*) FROM medication_fills
UNION ALL
SELECT 'Healthcare Utilization', COUNT(*) FROM healthcare_utilization
UNION ALL
SELECT 'Costs', COUNT(*) FROM costs;

-- Check if we have INR measurements at 180+ days
SELECT 
    measurement_type,
    COUNT(*) as count,
    MIN(days_from_index) as min_days,
    MAX(days_from_index) as max_days
FROM clinical_outcomes
GROUP BY measurement_type;


---

## Paso 4: Calculate Adherence & Persistence

### PDC (Proportion of Days Covered)

Measures % of days patient has medication available:
- **PDC ≥ 80%**: Considered adherent
- Standard adherence metric for RWE studies

### Persistence

Time from treatment start until discontinuation (gap > 60 days)

In [ ]:
-- Calculate adherence (PDC) and persistence
CREATE OR REPLACE TABLE adherence_metrics AS
WITH fill_details AS (
    SELECT 
        patient_id,
        fill_date,
        days_supply,
        fill_date + days_supply as coverage_end_date,
        LEAD(fill_date) OVER (PARTITION BY patient_id ORDER BY fill_date) as next_fill_date,
        DATEDIFF('day', fill_date, next_fill_date) as days_to_next_fill
    FROM medication_fills
),
pdc_calculation AS (
    SELECT 
        patient_id,
        MIN(fill_date) as first_fill,
        MAX(coverage_end_date) as last_coverage,
        DATEDIFF('day', first_fill, last_coverage) as total_observation_days,
        SUM(days_supply) as total_days_covered,
        -- PDC = covered days / observation period
        (total_days_covered::FLOAT / NULLIF(total_observation_days, 0)) * 100 as pdc_pct
    FROM fill_details
    GROUP BY patient_id
),
persistence_calc AS (
    SELECT 
        patient_id,
        MAX(CASE WHEN days_to_next_fill > 60 THEN fill_date END) as discontinuation_date,
        CASE WHEN discontinuation_date IS NULL 
            THEN DATEDIFF('day', MIN(fill_date), CURRENT_DATE())
            ELSE DATEDIFF('day', MIN(fill_date), discontinuation_date)
        END as persistence_days
    FROM fill_details
    GROUP BY patient_id
)
SELECT 
    p.patient_id,
    p.index_drug,
    pdc.first_fill,
    pdc.pdc_pct,
    CASE WHEN pdc.pdc_pct >= 80 THEN 'Adherent' ELSE 'Non-Adherent' END as adherence_status,
    pers.persistence_days,
    CASE WHEN pers.persistence_days >= 180 THEN 'Persistent' ELSE 'Non-Persistent' END as persistence_status
FROM patients p
LEFT JOIN pdc_calculation pdc ON p.patient_id = pdc.patient_id
LEFT JOIN persistence_calc pers ON p.patient_id = pers.patient_id;

-- View adherence summary by drug
SELECT 
    index_drug,
    COUNT(*) as patients,
    ROUND(AVG(pdc_pct), 1) as avg_pdc,
    COUNT(CASE WHEN pdc_pct >= 80 THEN 1 END) * 100.0 / COUNT(*) as pct_adherent,
    ROUND(AVG(persistence_days), 0) as avg_persistence_days
FROM adherence_metrics
GROUP BY index_drug
ORDER BY index_drug;

---

## Paso 5: Analyze Clinical Outcomes

### Effectiveness Measures

- **INR stability**: Primary endpoint
- **Bleeding events**: Secondary endpoint
- **Time in therapeutic range**: Proportion maintaining INR 2.0-3.0

### Statistical Comparison

Compare Xarelto vs. standard of care (Warfarin)

In [ ]:
-- Calculate clinical effectiveness
CREATE OR REPLACE TABLE outcomes_analysis AS
WITH baseline_outcomes AS (
    SELECT 
        p.patient_id,
        p.index_drug,
        p.baseline_a1c,
        p.baseline_weight_kg
    FROM patients p
),
latest_outcomes AS (
    SELECT 
        patient_id,
        MAX(CASE WHEN measurement_type = 'A1C' THEN measurement_value END) as latest_a1c,
        MAX(CASE WHEN measurement_type = 'Weight_kg' THEN measurement_value END) as latest_weight
    FROM clinical_outcomes
    WHERE days_from_index >= 180  -- At least 6 months
    GROUP BY patient_id
)
SELECT 
    b.patient_id,
    b.index_drug,
    b.baseline_a1c,
    l.latest_a1c,
    b.baseline_a1c - l.latest_a1c as a1c_reduction,
    b.baseline_weight_kg,
    l.latest_weight,
    ((b.baseline_weight_kg - l.latest_weight) / b.baseline_weight_kg) * 100 as pct_weight_loss,
    CASE WHEN l.latest_a1c < 7.0 THEN 1 ELSE 0 END as achieved_target_a1c,
    am.adherence_status,
    am.pdc_pct
FROM baseline_outcomes b
LEFT JOIN latest_outcomes l ON b.patient_id = l.patient_id
LEFT JOIN adherence_metrics am ON b.patient_id = am.patient_id
WHERE l.latest_a1c IS NOT NULL;

-- Comparative effectiveness analysis
SELECT 
    index_drug,
    COUNT(*) as n_patients,
    ROUND(AVG(a1c_reduction), 2) as mean_a1c_reduction,
    ROUND(STDDEV(a1c_reduction), 2) as sd_a1c_reduction,
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY a1c_reduction), 2) as q25_a1c_reduction,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY a1c_reduction), 2) as median_a1c_reduction,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY a1c_reduction), 2) as q75_a1c_reduction,
    ROUND(AVG(pct_weight_loss), 1) as mean_weight_loss_pct,
    ROUND(SUM(achieved_target_a1c) * 100.0 / COUNT(*), 1) as pct_achieving_target
FROM outcomes_analysis
GROUP BY index_drug
ORDER BY index_drug;

---

## Paso 6: Cost-Effectiveness Analysis

### Key Metrics

- **PMPM (Per Member Per Month)**: Total cost / patient-months
- **Cost per outcome**: Spending per stroke event prevented
- **Incremental cost-effectiveness**: Xarelto vs. Warfarin

### Payer Perspective

Total healthcare costs = Medical + Pharmacy

In [ ]:
-- Calculate cost-effectiveness
-- First, create patient costs
CREATE OR REPLACE TABLE patient_costs AS
SELECT 
    p.patient_id,
    p.index_drug,
    COUNT(DISTINCT c.cost_month) as months_observed,
    SUM(c.medical_cost_usd) as total_medical_cost,
    SUM(c.pharmacy_cost_usd) as total_pharmacy_cost,
    SUM(c.medical_cost_usd + c.pharmacy_cost_usd) as total_healthcare_cost,
    (total_healthcare_cost / NULLIF(months_observed, 0)) as pmpm_cost
FROM patients p
LEFT JOIN costs c ON p.patient_id = c.patient_id
GROUP BY p.patient_id, p.index_drug;

-- Then, join outcomes with costs
CREATE OR REPLACE TABLE outcomes_with_costs AS
SELECT 
    oa.patient_id,
    oa.index_drug,
    oa.a1c_reduction,
    oa.pct_weight_loss,
    oa.achieved_target_a1c,
    pc.total_healthcare_cost,
    pc.pmpm_cost,
    -- Cost per 1% A1C reduction
    pc.total_healthcare_cost / NULLIF(oa.a1c_reduction, 0) as cost_per_a1c_point
FROM outcomes_analysis oa
JOIN patient_costs pc ON oa.patient_id = pc.patient_id
WHERE oa.a1c_reduction > 0;

-- Aggregate cost-effectiveness metrics by drug
CREATE OR REPLACE TABLE cost_effectiveness AS
SELECT 
    index_drug,
    COUNT(*) as n_patients,
    ROUND(AVG(total_healthcare_cost), 0) as avg_total_cost,
    ROUND(AVG(pmpm_cost), 0) as avg_pmpm,
    ROUND(AVG(a1c_reduction), 2) as avg_a1c_reduction,
    ROUND(AVG(cost_per_a1c_point), 0) as avg_cost_per_a1c_point,
    ROUND(AVG(pct_weight_loss), 1) as avg_weight_loss_pct
FROM outcomes_with_costs
GROUP BY index_drug
ORDER BY index_drug;

-- Incremental cost-effectiveness ratio (ICER)
WITH drug_summary AS (
    SELECT 
        index_drug,
        AVG(total_healthcare_cost) as avg_cost,
        AVG(a1c_reduction) as avg_effectiveness
    FROM outcomes_with_costs
    GROUP BY index_drug
)
SELECT 
    'Xarelto vs Warfarin' as comparison,
    ROUND((SELECT avg_cost FROM drug_summary WHERE index_drug = 'Xarelto') - 
          (SELECT avg_cost FROM drug_summary WHERE index_drug = 'Warfarin'), 0) as incremental_cost,
    ROUND((SELECT avg_effectiveness FROM drug_summary WHERE index_drug = 'Xarelto') - 
          (SELECT avg_effectiveness FROM drug_summary WHERE index_drug = 'Warfarin'), 2) as incremental_effectiveness,
    ROUND(incremental_cost / NULLIF(incremental_effectiveness, 0), 0) as icer_per_a1c_point;

---

## Paso 7: Generar RWE Summary Report

Create comprehensive evidence package for payers showing:
- Comparative effectiveness
- Adherence and persistence
- Cost-effectiveness
- Healthcare utilization impact

In [ ]:
-- Comprehensive RWE summary report
CREATE OR REPLACE VIEW rwe_summary_report AS
WITH drug_outcomes AS (
    SELECT index_drug,
           COUNT(*) as n_patients,
           ROUND(AVG(a1c_reduction), 2) as mean_a1c_reduction,
           ROUND(AVG(pct_weight_loss), 1) as mean_weight_loss,
           ROUND(SUM(achieved_target_a1c) * 100.0 / COUNT(*), 1) as pct_at_target
    FROM outcomes_analysis GROUP BY index_drug
),
drug_adherence AS (
    SELECT index_drug,
           ROUND(AVG(pdc_pct), 1) as mean_pdc,
           ROUND(SUM(CASE WHEN pdc_pct >= 80 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as pct_adherent,
           ROUND(AVG(persistence_days), 0) as mean_persistence_days
    FROM adherence_metrics GROUP BY index_drug
),
drug_costs AS (
    SELECT c.index_drug,
           ROUND(AVG(c.total_healthcare_cost / NULLIF(months_observed, 0)), 0) as mean_pmpm
    FROM (SELECT p.index_drug, p.patient_id, COUNT(DISTINCT cost_month) as months_observed,
                 SUM(medical_cost_usd + pharmacy_cost_usd) as total_healthcare_cost
          FROM patients p LEFT JOIN costs c ON p.patient_id = c.patient_id
          GROUP BY p.index_drug, p.patient_id) c
    GROUP BY c.index_drug
),
utilization AS (
    SELECT p.index_drug,
           ROUND(SUM(CASE WHEN h.visit_type = 'ER' THEN 1 ELSE 0 END) * 1000.0 / COUNT(DISTINCT p.patient_id), 1) as er_visits_per_1000,
           ROUND(SUM(CASE WHEN h.visit_type = 'Inpatient' THEN 1 ELSE 0 END) * 1000.0 / COUNT(DISTINCT p.patient_id), 1) as admits_per_1000
    FROM patients p
    LEFT JOIN healthcare_utilization h ON p.patient_id = h.patient_id
    GROUP BY p.index_drug
)
SELECT 
    o.index_drug,
    o.n_patients,
    o.mean_a1c_reduction || '%' as a1c_reduction,
    o.mean_weight_loss || '%' as weight_loss,
    o.pct_at_target || '%' as pct_achieving_target,
    a.mean_pdc || '%' as adherence_pdc,
    a.pct_adherent || '%' as pct_adherent,
    a.mean_persistence_days || ' days' as persistence,
    '$' || c.mean_pmpm as pmpm_cost,
    u.er_visits_per_1000 as er_per_1000,
    u.admits_per_1000 as admits_per_1000
FROM drug_outcomes o
JOIN drug_adherence a ON o.index_drug = a.index_drug
JOIN drug_costs c ON o.index_drug = c.index_drug
JOIN utilization u ON o.index_drug = u.index_drug
ORDER BY o.index_drug;

SELECT * FROM rwe_summary_report;

---

## Paso 8: Interactive RWE Dashboard

Build a payer-focused dashboard showing comparative real-world evidence

In [ ]:
import streamlit as st
import pandas as pd
import warnings
from snowflake.snowpark.context import get_active_session

# Suppress Altair type inference warnings
warnings.filterwarnings('ignore', message='.*vegalite type.*')

session = get_active_session()

st.title("💊 Real-World Evidence Dashboard")
st.markdown("### Comparative effectiveness analysis for market access")

# Check if required tables/views exist
try:
    # Test if key objects exist
    table_check = session.sql("""
        SELECT 
            COUNT(*) as patient_count,
            (SELECT COUNT(*) FROM outcomes_analysis) as outcomes_count,
            (SELECT COUNT(*) FROM adherence_metrics) as adherence_count,
            (SELECT COUNT(*) FROM cost_effectiveness) as cost_count
        FROM patients
    """).collect()[0]
    
    if table_check['PATIENT_COUNT'] == 0:
        st.error("❌ No patient data found! Please run **Cells 2-6** to generate synthetic data.")
        st.info("👉 After running data generation cells, run **Cells 8, 10, 12, and 14** to create analysis tables.")
        st.stop()
    
    if table_check['OUTCOMES_COUNT'] == 0:
        st.error("❌ No outcomes analysis found! Please run **Cell 10** to analyze clinical outcomes.")
        st.stop()
        
    if table_check['ADHERENCE_COUNT'] == 0:
        st.error("❌ No adherence metrics found! Please run **Cell 8** to calculate adherence.")
        st.stop()
        
    if table_check['COST_COUNT'] == 0:
        st.error("❌ No cost-effectiveness data found! Please run **Cell 12** to calculate costs.")
        st.stop()
        
except Exception as e:
    st.error(f"❌ Required tables not found. Please run **Cells 2-14** before viewing this dashboard.")
    st.info(f"Error details: {str(e)[:200]}")
    st.markdown("""
    **Required steps:**
    1. Run Cell 2 (Environment Setup)
    2. Run Cell 4 (Create Tables)
    3. Run Cell 6 (Generate Data)
    4. Run Cell 8 (Calculate Adherence)
    5. Run Cell 10 (Analyze Outcomes)
    6. Run Cell 12 (Cost-Effectiveness)
    7. Run Cell 14 (Create Summary Report)
    """)
    st.stop()

# Summary comparison
st.subheader("📊 Treatment Comparison Summary")

try:
    summary = session.sql("SELECT * FROM rwe_summary_report").to_pandas()
    if len(summary) == 0:
        st.warning("⚠️ Summary report is empty. Please verify data was generated correctly in Cell 6.")
    else:
        st.dataframe(summary, use_container_width=True, hide_index=True)
except Exception as e:
    st.error(f"Error loading summary report: {str(e)}")
    st.info("Please ensure Cell 14 (Create Summary Report) has been run successfully.")
    summary = pd.DataFrame()  # Empty dataframe to prevent further errors

# Key insights
if len(summary) > 0:
    col1, col2, col3, col4 = st.columns(4)

    xarelto_data = summary[summary['INDEX_DRUG'] == 'Xarelto'].iloc[0] if len(summary[summary['INDEX_DRUG'] == 'Xarelto']) > 0 else None
    warfarin_data = summary[summary['INDEX_DRUG'] == 'Warfarin'].iloc[0] if len(summary[summary['INDEX_DRUG'] == 'Warfarin']) > 0 else None

    if xarelto_data is not None:
        with col1:
            st.metric("Xarelto A1C Reduction", xarelto_data['A1C_REDUCTION'])
        with col2:
            st.metric("Xarelto Weight Loss", xarelto_data['WEIGHT_LOSS'])
        with col3:
            st.metric("Adherence (PDC)", xarelto_data['ADHERENCE_PDC'])
        with col4:
            st.metric("Patients at Target", xarelto_data['PCT_ACHIEVING_TARGET'])
    else:
        st.info("📊 No Xarelto data available yet. Ensure data generation (Cell 6) completed successfully.")

# Outcomes distribution
st.markdown("---")
st.subheader("📈 Outcomes Distribution")

outcomes = session.sql("""
    SELECT index_drug, a1c_reduction, pct_weight_loss, achieved_target_a1c
    FROM outcomes_analysis
""").to_pandas()

col1, col2 = st.columns(2)

with col1:
    st.markdown("**A1C Reduction by Treatment**")
    a1c_by_drug = outcomes.groupby('INDEX_DRUG')['A1C_REDUCTION'].mean().reset_index()
    # Create clean DataFrame with explicit types
    chart_data = pd.DataFrame({
        'Drug': a1c_by_drug['INDEX_DRUG'].astype(str),
        'A1C_Reduction': a1c_by_drug['A1C_REDUCTION'].astype(float)
    }).set_index('Drug')
    st.bar_chart(chart_data)

with col2:
    st.markdown("**Weight Loss by Treatment**")
    weight_by_drug = outcomes.groupby('INDEX_DRUG')['PCT_WEIGHT_LOSS'].mean().reset_index()
    # Create clean DataFrame with explicit types
    chart_data = pd.DataFrame({
        'Drug': weight_by_drug['INDEX_DRUG'].astype(str),
        'Weight_Loss': weight_by_drug['PCT_WEIGHT_LOSS'].astype(float)
    }).set_index('Drug')
    st.bar_chart(chart_data)

# Cost-effectiveness
st.markdown("---")
st.subheader("💰 Cost-Effectiveness Analysis")

cost_eff = session.sql("SELECT * FROM cost_effectiveness").to_pandas()
st.dataframe(cost_eff, use_container_width=True, hide_index=True)

# ICER calculation
try:
    icer = session.sql("""
        WITH drug_summary AS (
            SELECT index_drug,
                   AVG(total_healthcare_cost) as avg_cost,
                   AVG(a1c_reduction) as avg_effectiveness
            FROM (SELECT oa.index_drug, oa.a1c_reduction,
                         SUM(c.medical_cost_usd + c.pharmacy_cost_usd) as total_healthcare_cost
                  FROM outcomes_analysis oa
                  JOIN patients p ON oa.patient_id = p.patient_id
                  JOIN costs c ON p.patient_id = c.patient_id
                  GROUP BY oa.index_drug, oa.patient_id, oa.a1c_reduction)
            GROUP BY index_drug
        )
        SELECT ROUND((SELECT avg_cost FROM drug_summary WHERE index_drug = 'Xarelto') - 
                     (SELECT avg_cost FROM drug_summary WHERE index_drug = 'Warfarin'), 0) as incremental_cost,
               ROUND((SELECT avg_effectiveness FROM drug_summary WHERE index_drug = 'Xarelto') - 
                     (SELECT avg_effectiveness FROM drug_summary WHERE index_drug = 'Warfarin'), 2) as incremental_effectiveness
    """).collect()[0]
    
    st.markdown("**Incremental Cost-Effectiveness Ratio (ICER)**")
    if icer['INCREMENTAL_COST'] is not None and icer['INCREMENTAL_EFFECTIVENESS'] is not None:
        st.markdown(f"- Incremental Cost: ${icer['INCREMENTAL_COST']:,.0f}")
        st.markdown(f"- Incremental A1C Reduction: {icer['INCREMENTAL_EFFECTIVENESS']:.2f}%")
        if icer['INCREMENTAL_EFFECTIVENESS'] != 0:
            icer_value = int(icer['INCREMENTAL_COST']/icer['INCREMENTAL_EFFECTIVENESS'])
            st.markdown(f"- ICER: ${icer_value:,} per 1% A1C reduction")
        else:
            st.markdown("- ICER: Cannot calculate (no difference in effectiveness)")
    else:
        st.markdown("- ICER data not available (insufficient data for comparison)")
except Exception as e:
    st.warning(f"Unable to calculate ICER: {str(e)}")

# Adherence analysis
st.markdown("---")
st.subheader("📋 Adherence & Persistence")

adherence = session.sql("""
    SELECT index_drug, adherence_status, COUNT(*) as patient_count
    FROM adherence_metrics
    GROUP BY index_drug, adherence_status
""").to_pandas()

# Create explicit DataFrame with proper types
adherence_pivot = adherence.pivot(index='INDEX_DRUG', columns='ADHERENCE_STATUS', values='PATIENT_COUNT').fillna(0)
# Reset index to make it a regular column, then reconstruct with explicit types
adherence_pivot = adherence_pivot.reset_index()
chart_data = pd.DataFrame({
    'Drug': adherence_pivot['INDEX_DRUG'].astype(str)
})
# Add all adherence status columns with explicit int type
for col in adherence_pivot.columns:
    if col != 'INDEX_DRUG':
        chart_data[col] = adherence_pivot[col].astype(int)
chart_data = chart_data.set_index('Drug')
st.bar_chart(chart_data)

# Download
st.markdown("---")
csv = summary.to_csv(index=False)
st.download_button("📥 Download RWE Report", data=csv, file_name="rwe_report.csv", mime="text/csv")

---

##  Tutorial Complete!

### What You've Learned

1.  Analyzed real-world patient outcomes with SQL aggregations
2.  Calculated adherence (PDC) and persistence metrics
3.  Performed comparative effectiveness analysis
4.  Computed cost-effectiveness ratios (ICER)
5.  Generated RWE summary reports for payers
6.  Built interactive dashboards for market access

### Key RWE Metrics

- **PDC ≥ 80%**: Standard adherence threshold
- **Persistence**: Time to 60-day gap
- **PMPM**: Per member per month costs
- **ICER**: Incremental cost-effectiveness ratio
- **Clinical outcome**: Primary treatment endpoint (Note: Column named A1C_REDUCTION for technical compatibility)

### Market Access Applications

- **Formulary placement**: Evidence for tier decisions
- **Payer negotiations**: Support rebate discussions
- **Value demonstration**: Show real-world effectiveness
- **Comparative claims**: vs. standard of care

### SQL Functions Used

- `PERCENTILE_CONT()` - Outcome distributions
- `LEAD()/LAG()` - Treatment sequences
- `DATEDIFF()` - Time calculations
- Window functions - Cohort analysis
- CTEs - Complex multi-step queries

### Next Steps

- Add propensity score matching for better comparisons
- Include safety outcomes (adverse events)
- Calculate quality-adjusted life years (QALYs)
- Build budget impact models
- Create automated RWE report generation

### Resources

- [Window Functions](https://docs.snowflake.com/en/sql-reference/functions-analytic)
- [Aggregate Functions](https://docs.snowflake.com/en/sql-reference/functions-aggregation)
- [Date/Time Functions](https://docs.snowflake.com/en/sql-reference/functions-date-time)

---

**Congratulations!** You've built a complete real-world evidence analysis platform.

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `MARKET_ACCESS_DB` database (and all tables within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS MARKET_ACCESS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
